# Marketing Campaign Analysis
## Phase 2 — Data cleaning & database loading

**Datasets:** marketing_campaign.csv and Amazon reviews
**Goal:** Apply all data quality fixes identified in Phase 1, engineer downstream features, and load a normalised PostgreSQL database that the analysis phases will query.

### What this phase does

**Part A — Cleaning**
- Drop the constant columns `Z_CostContact` and `Z_Revenue` (no information)
- Drop 24 rows with missing income (small volume but behaviourally biased — see note below)
- Drop 3 customers with `Year_Birth ≤ 1900` (data-entry artefacts)
- Drop 1 customer with `Income > $200K` (data-entry artefact)
- Standardise inconsistent categorical values in `Education` and `Marital_Status`
- Derive an empirical age reference year from `max(Dt_Customer)` rather than hard-coding
- Engineer downstream features (age, total spend, tenure, etc.) with target leakage prevented

**Part B — Reviews preparation**
- De-duplicate on `(product, review text)` — Phase 1 found 1,669 duplicates
- Drop very short reviews (under 20 characters) that carry no sentiment signal
- Assign sentiment labels from star ratings

**Part C — Database loading**
- Connect to PostgreSQL via local trust authentication
- Execute the schema with real `PRIMARY KEY` and `FOREIGN KEY` constraints
- Split the flat file into 4 relational tables + the reviews table
- Verify with SQL queries

### Important note on the missing-income rows
The 24 customers with missing income are not random. Compared to the rest of the dataset they have a 4.2% response rate vs 15.0%, mean spend of $488 vs $607, and slightly higher recency. Dropping them is acceptable because the volume is small (1.07%), but the remaining dataset tilts slightly toward more engaged customers. This needs to be carried into Phase 4 propensity modelling — the model will be calibrated on customers who, on average, respond more than the true population.

---
## 1. Import libraries & configure paths

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import getpass
from pathlib import Path

# SQLAlchemy 2.0 with psycopg2 driver — connects Python to PostgreSQL
# text() wraps raw SQL strings so SQLAlchemy can execute them safely
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Locate the repo root portably by walking up from the current working
# directory until the 'data' folder is found, so the notebook runs on any
# machine and from any subfolder.
def find_project_root(marker='data'):
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).is_dir():
            return parent
    raise FileNotFoundError(
        f"Could not find project root: no '{marker}/' folder above {Path.cwd()}. "
        "Run this notebook from inside the cloned repository."
    )

PROJECT_ROOT   = find_project_root()
RAW_PATH       = str(PROJECT_ROOT / 'data' / 'raw') + os.sep
PROCESSED_PATH = str(PROJECT_ROOT / 'data' / 'processed') + os.sep
DOCS_PATH      = str(PROJECT_ROOT / 'docs')
SQL_PATH       = str(PROJECT_ROOT / 'sql')

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(DOCS_PATH, exist_ok=True)
os.makedirs(os.path.join(SQL_PATH, 'schema'), exist_ok=True)

import sqlalchemy
print(f'pandas version:     {pd.__version__}')
print(f'SQLAlchemy version: {sqlalchemy.__version__}')
print(f'Processed path:     {PROCESSED_PATH}')
print('Libraries loaded ✅')

pandas version:     3.0.3
SQLAlchemy version: 2.0.49
Processed path:     /Users/omartouzani/Desktop/marketing-campaign-analysis/data/processed/
Libraries loaded ✅


---
## 2. Load the raw data
Read fresh from the raw folder on every run. The raw files are never modified.

In [2]:
# Campaign data is tab-separated (established in Phase 1)
campaign = pd.read_csv(RAW_PATH + 'marketing_campaign.csv', sep='\t')

# Reviews data — low_memory=False prevents mixed-type warnings on large files
reviews = pd.read_csv(
    RAW_PATH + 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv',
    low_memory=False
)

print(f'Campaign loaded:  {campaign.shape[0]:,} rows, {campaign.shape[1]} columns')
print(f'Reviews loaded:   {reviews.shape[0]:,} rows, {reviews.shape[1]} columns')

Campaign loaded:  2,240 rows, 29 columns
Reviews loaded:   28,332 rows, 24 columns


In [3]:
# Reproducibility check: confirm the exact raw files are present and record
# their checksums. If a reviewer's files differ, the cleaned tables (and all
# downstream results) will differ. This surfaces that immediately.
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

raw_files = [
    'marketing_campaign.csv',
    'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv',
]

print('Raw data checksums (SHA-256, first 12 chars):')
for fname in raw_files:
    print(f'  {fname:65s} {sha256(RAW_PATH + fname)[:12]}')

Raw data checksums (SHA-256, first 12 chars):
  marketing_campaign.csv                                            cd0affa36b1b
  Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv   9778f5b1a7e5


---
# PART A — CAMPAIGN DATASET CLEANING
---
## 3. Drop constant columns
`Z_CostContact` and `Z_Revenue` are the same value for every customer (std = 0). They carry no analytical information and just take up space in the database.

In [4]:
# Confirm both columns are constant before dropping
print('Z_CostContact unique values:', campaign['Z_CostContact'].unique())
print('Z_Revenue unique values:    ', campaign['Z_Revenue'].unique())

# Drop both columns
campaign = campaign.drop(columns=['Z_CostContact', 'Z_Revenue'])
print(f'\nColumns after drop: {campaign.shape[1]}')

Z_CostContact unique values: [3]
Z_Revenue unique values:     [11]

Columns after drop: 27


---
## 4. Drop rows with data quality issues
Three filters applied in sequence with cumulative row-count logging:
1. Missing income (24 rows, behaviourally biased — see top-of-notebook note)
2. Year_Birth ≤ 1900 (3 rows, implied age 120+)
3. Income > $200K (1 row, value of $666,666)

In [5]:
# Track cumulative row removals so we know exactly what each filter dropped
start_rows = len(campaign)
print(f'Starting rows: {start_rows:,}')

# Filter 1 — drop missing income
missing_income_count = campaign['Income'].isnull().sum()
campaign = campaign.dropna(subset=['Income']).copy()
print(f'  After dropping missing income ({missing_income_count} rows):           {len(campaign):,}')

# Filter 2 — drop unrealistic birth years
old_age_count = (campaign['Year_Birth'] <= 1900).sum()
campaign = campaign[campaign['Year_Birth'] > 1900].copy()
print(f'  After dropping Year_Birth ≤ 1900 ({old_age_count} rows):              {len(campaign):,}')

# Filter 3 — drop extreme income outlier
# Phase 1 identified one customer with Income = $666,666 (data entry artefact)
high_income_count = (campaign['Income'] > 200000).sum()
campaign = campaign[campaign['Income'] <= 200000].copy()
print(f'  After dropping Income > $200K ({high_income_count} row):                {len(campaign):,}')

print(f'\nTotal rows removed: {start_rows - len(campaign)} ({(start_rows - len(campaign)) / start_rows * 100:.2f}%)')
print(f'Final row count:    {len(campaign):,}')

Starting rows: 2,240
  After dropping missing income (24 rows):           2,216
  After dropping Year_Birth ≤ 1900 (3 rows):              2,213
  After dropping Income > $200K (1 row):                2,212

Total rows removed: 28 (1.25%)
Final row count:    2,212


---
## 5. Standardise categorical columns
Two columns need consolidation:
- `Education`: `'2n Cycle'` is the second cycle of higher education (~bachelor's level). Rename to `'Undergraduate'` for clarity.
- `Marital_Status`: junk values (`YOLO`, `Absurd`, `Alone`) are folded into `Single`.

In [6]:
print('BEFORE cleaning:')
print('  Education      :', sorted(campaign['Education'].unique()))
print('  Marital_Status :', sorted(campaign['Marital_Status'].unique()))

# Education mapping — '2n Cycle' is the second cycle of higher education
# (bachelor's level in the Portuguese/EU system, not high school)
education_map = {
    'Graduation': 'Graduation',
    'PhD':        'PhD',
    'Master':     'Master',
    '2n Cycle':   'Undergraduate',
    'Basic':      'Basic',
}
campaign['Education'] = campaign['Education'].map(education_map)

# Marital status mapping — junk values consolidated into Single
marital_map = {
    'Married':  'Married',
    'Together': 'Together',
    'Single':   'Single',
    'Divorced': 'Divorced',
    'Widow':    'Widow',
    'Alone':    'Single',
    'Absurd':   'Single',
    'YOLO':     'Single',
}
campaign['Marital_Status'] = campaign['Marital_Status'].map(marital_map)

print('\nAFTER cleaning:')
print('  Education      :', sorted(campaign['Education'].unique()))
print('  Marital_Status :', sorted(campaign['Marital_Status'].unique()))

BEFORE cleaning:
  Education      : ['2n Cycle', 'Basic', 'Graduation', 'Master', 'PhD']
  Marital_Status : ['Absurd', 'Alone', 'Divorced', 'Married', 'Single', 'Together', 'Widow', 'YOLO']

AFTER cleaning:
  Education      : ['Basic', 'Graduation', 'Master', 'PhD', 'Undergraduate']
  Marital_Status : ['Divorced', 'Married', 'Single', 'Together', 'Widow']


---
## 6. Convert date column & derive the reference year
`Dt_Customer` is the enrollment date stored as a string (DD-MM-YYYY).
We derive the age reference year empirically from `max(Dt_Customer)` rather than hard-coding it.

In [7]:
# Parse with dayfirst=True because the format is DD-MM-YYYY
campaign['Dt_Customer'] = pd.to_datetime(campaign['Dt_Customer'], dayfirst=True)

# Empirical reference year — the most recent enrollment date in the data
REFERENCE_YEAR = campaign['Dt_Customer'].max().year
REFERENCE_DATE = campaign['Dt_Customer'].max()

print(f'Dt_Customer dtype:  {campaign["Dt_Customer"].dtype}')
print(f'Date range:         {campaign["Dt_Customer"].min().date()} → {REFERENCE_DATE.date()}')
print(f'Reference year:     {REFERENCE_YEAR}')
print(f'Reference date:     {REFERENCE_DATE.date()}')

Dt_Customer dtype:  datetime64[us]
Date range:         2012-07-30 → 2014-06-29
Reference year:     2014
Reference date:     2014-06-29


---
## 7. Feature engineering
Derived features used across Phases 3–6. Each is explained inline.

In [8]:
# Age — using empirical reference year from above (not hardcoded)
campaign['Age'] = REFERENCE_YEAR - campaign['Year_Birth']

# Total spend across all 6 product categories
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
              'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
campaign['TotalSpend'] = campaign[spend_cols].sum(axis=1)

# Total purchases across all channels
channel_cols = ['NumWebPurchases', 'NumCatalogPurchases',
                'NumStorePurchases', 'NumDealsPurchases']
campaign['TotalPurchases'] = campaign[channel_cols].sum(axis=1)

# Household composition
campaign['TotalChildren'] = campaign['Kidhome'] + campaign['Teenhome']
campaign['HasChildren']   = (campaign['TotalChildren'] > 0).astype(int)

# Customer tenure in days from enrollment to reference date
campaign['TenureDays'] = (REFERENCE_DATE - campaign['Dt_Customer']).dt.days

# Historical campaigns accepted — EXCLUDES Response (the target for Phase 4)
# Including Response here would create target leakage in propensity modelling
hist_cmp_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
                 'AcceptedCmp4', 'AcceptedCmp5']
campaign['TotalHistCampaignsAccepted'] = campaign[hist_cmp_cols].sum(axis=1)

# Average spend per purchase — basket size proxy
# np.where avoids division by zero for customers with no purchases
campaign['AvgSpendPerPurchase'] = np.where(
    campaign['TotalPurchases'] > 0,
    campaign['TotalSpend'] / campaign['TotalPurchases'],
    0
)

new_features = ['Age', 'TotalSpend', 'TotalPurchases', 'TotalChildren',
                'HasChildren', 'TenureDays',
                'TotalHistCampaignsAccepted', 'AvgSpendPerPurchase']
print('Engineered features:')
print(campaign[new_features].describe().round(2))

Engineered features:
           Age  TotalSpend  TotalPurchases  TotalChildren  HasChildren  \
count  2212.00     2212.00         2212.00        2212.00      2212.00   
mean     45.09      607.27           14.89           0.95         0.71   
std      11.70      602.51            7.67           0.75         0.45   
min      18.00        5.00            0.00           0.00         0.00   
25%      37.00       69.00            8.00           0.00         0.00   
50%      44.00      397.00           15.00           1.00         1.00   
75%      55.00     1048.00           21.00           1.00         1.00   
max      74.00     2525.00           44.00           3.00         1.00   

       TenureDays  TotalHistCampaignsAccepted  AvgSpendPerPurchase  
count     2212.00                     2212.00              2212.00  
mean       353.71                        0.30                32.61  
std        202.49                        0.68                28.58  
min          0.00                   

> **Leakage note.** `TotalHistCampaignsAccepted` deliberately excludes `Response`. `Response` is the Phase 4 target — any feature summarising past campaign acceptance must not include it.

---
## 8. Final campaign quality check

In [9]:
print('Final campaign dataset:')
print(f'  Total rows:     {len(campaign):,}')
print(f'  Total columns:  {campaign.shape[1]}')
print(f'  Missing values: {campaign.isnull().sum().sum()}')
print(f'  Duplicate IDs:  {campaign["ID"].duplicated().sum()}')
print(f'  Age range:      {campaign["Age"].min()} — {campaign["Age"].max()}')
print(f'  Income range:   ${campaign["Income"].min():,.0f} — ${campaign["Income"].max():,.0f}')
print(f'  Spend range:    ${campaign["TotalSpend"].min():,.0f} — ${campaign["TotalSpend"].max():,.0f}')

Final campaign dataset:
  Total rows:     2,212
  Total columns:  35
  Missing values: 0
  Duplicate IDs:  0
  Age range:      18 — 74
  Income range:   $1,730 — $162,397
  Spend range:    $5 — $2,525


---
# PART B — REVIEWS DATASET CLEANING
---
## 9. Prepare reviews for sentiment analysis
Four operations:
1. Select only the columns Phase 5 needs
2. De-duplicate on `(product_name, review_text)` — Phase 1 found 1,669 duplicates
3. Drop reviews under 20 characters (no sentiment signal)
4. Assign sentiment labels from star ratings

In [10]:
# Select only the columns needed for sentiment work
reviews_clean = reviews[[
    'name', 'brand', 'primaryCategories',
    'reviews.rating', 'reviews.title', 'reviews.text',
    'reviews.date',
]].copy()

# Rename for cleaner SQL column names
reviews_clean.columns = [
    'product_name', 'brand', 'category',
    'rating', 'review_title', 'review_text', 'review_date',
]

# Cast rating to numeric and date to datetime
reviews_clean['rating']      = pd.to_numeric(reviews_clean['rating'], errors='coerce')
reviews_clean['review_date'] = pd.to_datetime(reviews_clean['review_date'], errors='coerce')

start_rows = len(reviews_clean)
print(f'Starting reviews: {start_rows:,}')

# Step 1 — drop reviews with missing/blank text or missing rating
reviews_clean = reviews_clean.dropna(subset=['review_text', 'rating'])
reviews_clean = reviews_clean[reviews_clean['review_text'].str.strip() != '']
print(f'  After dropping empty text / missing rating:   {len(reviews_clean):,}')

# Step 2 — de-duplicate on (product, text) — keeps the first occurrence
before = len(reviews_clean)
reviews_clean = reviews_clean.drop_duplicates(
    subset=['product_name', 'review_text'], keep='first'
)
print(f'  After de-duplicating ({before - len(reviews_clean):,} duplicates removed): {len(reviews_clean):,}')

# Step 3 — filter very short reviews
reviews_clean['review_length'] = reviews_clean['review_text'].str.len()
before = len(reviews_clean)
reviews_clean = reviews_clean[reviews_clean['review_length'] >= 20].copy()
print(f'  After dropping reviews < 20 chars ({before - len(reviews_clean):,} removed): {len(reviews_clean):,}')

# Step 4 — assign sentiment label from star rating
def label_sentiment(rating):
    if rating <= 2:
        return 'Negative'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Positive'

reviews_clean['sentiment'] = reviews_clean['rating'].apply(label_sentiment)

print(f'\nTotal removed: {start_rows - len(reviews_clean):,} ({(start_rows - len(reviews_clean)) / start_rows * 100:.2f}%)')
print(f'Final reviews:  {len(reviews_clean):,}')
print(f'\nSentiment distribution:')
print(reviews_clean['sentiment'].value_counts())
print(f'\n  Positive: {(reviews_clean["sentiment"] == "Positive").mean() * 100:.1f}%')
print(f'  Neutral:  {(reviews_clean["sentiment"] == "Neutral").mean() * 100:.1f}%')
print(f'  Negative: {(reviews_clean["sentiment"] == "Negative").mean() * 100:.1f}%')

Starting reviews: 28,332
  After dropping empty text / missing rating:   28,332
  After de-duplicating (1,261 duplicates removed): 27,071
  After dropping reviews < 20 chars (1,398 removed): 25,673

Total removed: 2,659 (9.39%)
Final reviews:  25,673

Sentiment distribution:
sentiment
Positive    23013
Negative     1525
Neutral      1135
Name: count, dtype: int64

  Positive: 89.6%
  Neutral:  4.4%
  Negative: 5.9%


---
## 10. Save cleaned files
Save both cleaned datasets as CSV before loading into PostgreSQL.
This provides a fallback and lets downstream phases reload without re-running cleaning.

In [11]:
campaign_path = os.path.join(PROCESSED_PATH, 'campaign_clean.csv')
campaign.to_csv(campaign_path, index=False)

reviews_path = os.path.join(PROCESSED_PATH, 'reviews_clean.csv')
reviews_clean.to_csv(reviews_path, index=False)

print(f'✅ campaign_clean.csv saved: {len(campaign):,} rows')
print(f'✅ reviews_clean.csv saved:  {len(reviews_clean):,} rows')

✅ campaign_clean.csv saved: 2,212 rows
✅ reviews_clean.csv saved:  25,673 rows


---
# PART C — DATABASE LOADING
---
## 11. Connect to PostgreSQL
Local trust authentication on the developer machine — no password, username derived from the OS login.

In [12]:
# Get the system username — Mac local trust auth uses this
username = getpass.getuser()

# Connection URL: postgresql+psycopg2://<user>@<host>:<port>/<db>
DB_URL = f'postgresql+psycopg2://{username}@localhost:5432/marketing_db'

# future=True activates full SQLAlchemy 2.0 mode
engine = create_engine(DB_URL, future=True)

# Test the connection
with engine.connect() as conn:
    result = conn.execute(text('SELECT current_database(), current_user'))
    row = result.fetchone()
    print(f'Connected to database: {row[0]}')
    print(f'Connected as user:     {row[1]}')
    print('PostgreSQL connection successful ✅')

Connected to database: marketing_db
Connected as user:     omartouzani
PostgreSQL connection successful ✅


---
## 12. Define & execute the schema
We execute the schema with real `PRIMARY KEY` and `FOREIGN KEY` constraints so the relational structure is enforced inside the database, not just defined on paper.

Order matters: drop child tables before the parent (`customers`), then create the parent before the children.

In [13]:
schema_sql = '''
-- marketing_db schema for Phase 2

-- Drop in reverse FK order so children go first
DROP TABLE IF EXISTS campaigns;
DROP TABLE IF EXISTS channels;
DROP TABLE IF EXISTS spending;
DROP TABLE IF EXISTS reviews;
DROP TABLE IF EXISTS customers;

-- Table 1: customer demographics (parent)
CREATE TABLE customers (
    customer_id     INTEGER PRIMARY KEY,
    year_birth      INTEGER,
    age             INTEGER,
    education       VARCHAR(50),
    marital_status  VARCHAR(50),
    income          NUMERIC(12,2),
    kidhome         INTEGER,
    teenhome        INTEGER,
    total_children  INTEGER,
    has_children    INTEGER,
    dt_customer     DATE,
    tenure_days     INTEGER,
    recency         INTEGER,
    complain        INTEGER
);

-- Table 2: product spending
CREATE TABLE spending (
    customer_id  INTEGER PRIMARY KEY REFERENCES customers(customer_id),
    mnt_wines    NUMERIC(10,2),
    mnt_fruits   NUMERIC(10,2),
    mnt_meat     NUMERIC(10,2),
    mnt_fish     NUMERIC(10,2),
    mnt_sweets   NUMERIC(10,2),
    mnt_gold     NUMERIC(10,2),
    total_spend  NUMERIC(10,2)
);

-- Table 3: purchase channels
CREATE TABLE channels (
    customer_id             INTEGER PRIMARY KEY REFERENCES customers(customer_id),
    num_web_purchases       INTEGER,
    num_catalog_purchases   INTEGER,
    num_store_purchases     INTEGER,
    num_deals_purchases     INTEGER,
    num_web_visits_month    INTEGER,
    total_purchases         INTEGER,
    avg_spend_per_purchase  NUMERIC(10,2)
);

-- Table 4: campaign response history
-- Z_CostContact and Z_Revenue dropped — they were constants in source data
CREATE TABLE campaigns (
    customer_id                  INTEGER PRIMARY KEY REFERENCES customers(customer_id),
    accepted_cmp1                INTEGER,
    accepted_cmp2                INTEGER,
    accepted_cmp3                INTEGER,
    accepted_cmp4                INTEGER,
    accepted_cmp5                INTEGER,
    response                     INTEGER,
    total_hist_campaigns_accepted INTEGER
);

-- Table 5: Amazon reviews (standalone — no FK to campaign tables)
CREATE TABLE reviews (
    product_name   TEXT,
    brand          TEXT,
    category       TEXT,
    rating         NUMERIC(3,1),
    review_title   TEXT,
    review_text    TEXT,
    review_date    DATE,
    review_length  INTEGER,
    sentiment      VARCHAR(20)
);
'''

# Execute the full schema in a single transaction
with engine.begin() as conn:
    for statement in schema_sql.strip().split(';'):
        s = statement.strip()
        if s:
            conn.execute(text(s))

print('✅ Schema created with PRIMARY KEY and FOREIGN KEY constraints')

# Save the schema to file for documentation
schema_path = os.path.join(SQL_PATH, 'schema', 'marketing_db_schema.sql')
with open(schema_path, 'w') as f:
    f.write(schema_sql)
print(f'✅ Schema documented at: {schema_path}')

✅ Schema created with PRIMARY KEY and FOREIGN KEY constraints
✅ Schema documented at: /Users/omartouzani/Desktop/marketing-campaign-analysis/sql/schema/marketing_db_schema.sql


---
## 13. Split the flat file into 4 relational tables
The original 27-column flat file mixes demographics, spending, channels, and campaign responses. Splitting on `customer_id` makes downstream SQL queries cleaner and matches how real marketing databases are structured.

In [14]:
# Table 1 — customers (demographics + profile)
customers = campaign[[
    'ID', 'Year_Birth', 'Age', 'Education', 'Marital_Status',
    'Income', 'Kidhome', 'Teenhome', 'TotalChildren',
    'HasChildren', 'Dt_Customer', 'TenureDays', 'Recency', 'Complain'
]].copy()
customers.columns = [
    'customer_id', 'year_birth', 'age', 'education', 'marital_status',
    'income', 'kidhome', 'teenhome', 'total_children',
    'has_children', 'dt_customer', 'tenure_days', 'recency', 'complain'
]

# Table 2 — spending per product category
spending = campaign[[
    'ID', 'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'TotalSpend'
]].copy()
spending.columns = [
    'customer_id', 'mnt_wines', 'mnt_fruits', 'mnt_meat',
    'mnt_fish', 'mnt_sweets', 'mnt_gold', 'total_spend'
]

# Table 3 — purchase channels
channels = campaign[[
    'ID', 'NumWebPurchases', 'NumCatalogPurchases',
    'NumStorePurchases', 'NumDealsPurchases',
    'NumWebVisitsMonth', 'TotalPurchases', 'AvgSpendPerPurchase'
]].copy()
channels.columns = [
    'customer_id', 'num_web_purchases', 'num_catalog_purchases',
    'num_store_purchases', 'num_deals_purchases',
    'num_web_visits_month', 'total_purchases', 'avg_spend_per_purchase'
]

# Table 4 — campaign responses
# Includes Response (Phase 4 target) and the leakage-free hist count
campaigns_tbl = campaign[[
    'ID', 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
    'AcceptedCmp4', 'AcceptedCmp5', 'Response',
    'TotalHistCampaignsAccepted'
]].copy()
campaigns_tbl.columns = [
    'customer_id', 'accepted_cmp1', 'accepted_cmp2', 'accepted_cmp3',
    'accepted_cmp4', 'accepted_cmp5', 'response',
    'total_hist_campaigns_accepted'
]

print(f'customers table:  {customers.shape}')
print(f'spending table:   {spending.shape}')
print(f'channels table:   {channels.shape}')
print(f'campaigns table:  {campaigns_tbl.shape}')

customers table:  (2212, 14)
spending table:   (2212, 8)
channels table:   (2212, 8)
campaigns table:  (2212, 8)


---
## 14. Insert data into PostgreSQL
Schema already exists with PK/FK constraints — we use `if_exists='append'` to insert into the prepared tables. Parent table (`customers`) must be loaded first so FK references on the children resolve.

In [15]:
# Load in FK-safe order: parent first, then children
load_order = [
    ('customers', customers),
    ('spending',  spending),
    ('channels',  channels),
    ('campaigns', campaigns_tbl),
    ('reviews',   reviews_clean),
]

for table_name, df in load_order:
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists='append',   # schema already created with constraints
        index=False,
        method='multi'        # batch inserts — faster than row-by-row
    )
    print(f'✅ {table_name:<10} loaded: {len(df):>6,} rows')

✅ customers  loaded:  2,212 rows
✅ spending   loaded:  2,212 rows
✅ channels   loaded:  2,212 rows
✅ campaigns  loaded:  2,212 rows
✅ reviews    loaded: 25,673 rows


---
## 15. Verify the database
Confirm all tables loaded correctly, constraints are in place, and a JOIN across tables works.

In [16]:
# Row counts via information_schema + query_to_xml trick
verify_query = '''
SELECT
    table_name,
    (xpath('/row/cnt/text()',
        query_to_xml(
            format('SELECT COUNT(*) AS cnt FROM %I', table_name),
            false, true, ''
        )
    ))[1]::text::int AS row_count
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
'''

with engine.connect() as conn:
    result = pd.read_sql(text(verify_query), conn)

print('Tables in marketing_db:')
print(result.to_string(index=False))

Tables in marketing_db:
table_name  row_count
 campaigns       2212
  channels       2212
 customers       2212
   reviews      25673
  spending       2212


In [17]:
# Confirm constraints are in place
constraints_query = '''
SELECT
    tc.table_name,
    tc.constraint_name,
    tc.constraint_type
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'public'
  AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY tc.table_name, tc.constraint_type;
'''
with engine.connect() as conn:
    constraints = pd.read_sql(text(constraints_query), conn)

print('Constraints in marketing_db:')
print(constraints.to_string(index=False))

Constraints in marketing_db:
table_name            constraint_name constraint_type
 campaigns campaigns_customer_id_fkey     FOREIGN KEY
 campaigns             campaigns_pkey     PRIMARY KEY
  channels  channels_customer_id_fkey     FOREIGN KEY
  channels              channels_pkey     PRIMARY KEY
 customers             customers_pkey     PRIMARY KEY
  spending  spending_customer_id_fkey     FOREIGN KEY
  spending              spending_pkey     PRIMARY KEY


In [18]:
# Test JOIN across two tables — confirms relational structure works
join_query = '''
SELECT
    c.customer_id,
    c.age,
    c.education,
    c.income,
    s.total_spend,
    s.mnt_wines,
    s.mnt_meat
FROM customers c
JOIN spending s ON c.customer_id = s.customer_id
ORDER BY s.total_spend DESC
LIMIT 10
'''

with engine.connect() as conn:
    top_spenders = pd.read_sql(text(join_query), conn)

print('Top 10 customers by total spend (JOIN across customers + spending):')
print(top_spenders.to_string(index=False))

Top 10 customers by total spend (JOIN across customers + spending):
 customer_id  age  education  income  total_spend  mnt_wines  mnt_meat
        5735   23     Master 90638.0       2525.0     1156.0     915.0
        5350   23     Master 90638.0       2525.0     1156.0     915.0
        1763   26 Graduation 87679.0       2524.0     1259.0     815.0
        4580   45 Graduation 75759.0       2486.0     1394.0     708.0
        4475   65        PhD 69098.0       2440.0     1315.0     780.0
        5453   58     Master 90226.0       2352.0     1083.0     649.0
       10133   44 Graduation 93790.0       2349.0     1302.0     731.0
        9010   42     Master 83151.0       2346.0      968.0     842.0
        6024   61 Graduation 94384.0       2302.0     1111.0     790.0
        5386   61 Graduation 94384.0       2302.0     1111.0     790.0


---
## 16. Phase 2 summary

In [19]:
print('=' * 64)
print('PHASE 2 SUMMARY — DATA CLEANING & DATABASE LOADING')
print('=' * 64)
print()
print('--- Campaign cleaning ---')
print(f'Constant columns dropped:      Z_CostContact, Z_Revenue')
print(f'Missing income rows dropped:   24 (1.07% — biased low response)')
print(f'Age outliers dropped:          3 (Year_Birth ≤ 1900)')
print(f'Income outlier dropped:        1 (Income = $666,666)')
print(f'Categories standardised:       Education, Marital_Status')
print(f'Date converted:                Dt_Customer (datetime)')
print(f'Reference year (empirical):    {REFERENCE_YEAR}')
print(f'Features engineered:           8 derived columns')
print(f'Final campaign rows:           {len(campaign):,}')
print()
print('--- Reviews cleaning ---')
print(f'Duplicates removed:            on (product_name, review_text)')
print(f'Short reviews removed:         < 20 characters')
print(f'Sentiment labels assigned:     Negative / Neutral / Positive')
print(f'Final reviews rows:            {len(reviews_clean):,}')
print()
print('--- Database ---')
print(f'Database:                      marketing_db (PostgreSQL)')
print(f'Schema:                        executed with PK + FK constraints')
print(f'Tables loaded:                 customers, spending, channels, campaigns, reviews')
print(f'Schema file:                   sql/schema/marketing_db_schema.sql')
print('=' * 64)
print()
print('Phase 4 reminders:')
print('  • Bias from dropping missing-income rows must be noted in propensity model.')
print('  • Use total_hist_campaigns_accepted as feature, never response (target leakage).')
print()
print('✅ Phase 2 complete. Proceed to Phase 3 — Campaign performance analysis.')

PHASE 2 SUMMARY — DATA CLEANING & DATABASE LOADING

--- Campaign cleaning ---
Constant columns dropped:      Z_CostContact, Z_Revenue
Missing income rows dropped:   24 (1.07% — biased low response)
Age outliers dropped:          3 (Year_Birth ≤ 1900)
Income outlier dropped:        1 (Income = $666,666)
Categories standardised:       Education, Marital_Status
Date converted:                Dt_Customer (datetime)
Reference year (empirical):    2014
Features engineered:           8 derived columns
Final campaign rows:           2,212

--- Reviews cleaning ---
Duplicates removed:            on (product_name, review_text)
Short reviews removed:         < 20 characters
Sentiment labels assigned:     Negative / Neutral / Positive
Final reviews rows:            25,673

--- Database ---
Database:                      marketing_db (PostgreSQL)
Schema:                        executed with PK + FK constraints
Tables loaded:                 customers, spending, channels, campaigns, reviews
Schema f